# Predictive PC-FMCW/DPSK — WOMD paper pipeline

This notebook downloads official WOMD v1.3.1 scenario shards, builds causal true-SDC samples, runs the four-objective/three-seed training ablation, and saves recoverable outputs to Google Drive.

Run the cells in order. Start with `RUN_MODE = "smoke"`; after it completes, switch to `"paper"` and run again.


In [ ]:
# 1. Google authentication and persistent Drive output
from google.colab import auth, drive

auth.authenticate_user()
drive.mount("/content/drive")
print("Google authentication and Drive mount: OK")


In [ ]:
# 2. Configuration and GPU verification
from pathlib import Path
import os, shutil, subprocess, sys
import torch

RUN_MODE = "smoke"  # change to "paper" only after the smoke run succeeds
assert RUN_MODE in {"smoke", "paper"}
SMOKE = RUN_MODE == "smoke"

ROOT = Path("/content/predictive-pc-fmcw")
DATA = Path("/content/womd")
DRIVE_OUT = Path("/content/drive/MyDrive/predictive_pc_fmcw_paper")
DATA.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run from the first cell.")


In [ ]:
# 3. Clone the exact repository and install Python-3.13-compatible dependencies
# The repository name itself ends in '.', therefore the clone URL has '..git'.
REPOSITORY_URL = "https://github.com/panagiotagrosdouli/predictive-pc-fmcw-vehicular-communications..git"

if ROOT.exists() and not (ROOT / ".git").exists():
    shutil.rmtree(ROOT)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "pip", "setuptools>=75", "wheel", "packaging"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy>=2.0", "scipy>=1.14", "matplotlib>=3.9", "protobuf>=5"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[ml]",
                "--no-build-isolation"], check=True)

# Colab may not refresh editable-install paths inside the already-running kernel.
# Add the repository src directory explicitly and invalidate import caches.
import importlib
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()

import predictive_pc_fmcw
print("Repository and project import: OK")
print("Imported from:", predictive_pc_fmcw.__file__)


In [ ]:
# 4. Download isolated official WOMD training and validation shards
BUCKET = "gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario"
TRAIN_SHARDS = 1 if SMOKE else 50
VALIDATION_SHARDS = 1 if SMOKE else 40
train_dir = DATA / "training"
validation_dir = DATA / "validation"
train_dir.mkdir(parents=True, exist_ok=True)
validation_dir.mkdir(parents=True, exist_ok=True)

def download_split(split, count, total, destination):
    for i in range(count):
        name = f"{split}.tfrecord-{i:05d}-of-{total:05d}"
        target = destination / name
        if not target.exists():
            subprocess.run(["gcloud", "storage", "cp", f"{BUCKET}/{split}/{name}", str(target)], check=True)

download_split("training", TRAIN_SHARDS, 1000, train_dir)
download_split("validation", VALIDATION_SHARDS, 150, validation_dir)
print("Training shards:", len(list(train_dir.glob("training.tfrecord-*"))))
print("Validation shards:", len(list(validation_dir.glob("validation.tfrecord-*"))))


In [ ]:
# 5. Build causal true-SDC training samples (no TensorFlow/Waymo wheel required)
train_npz = DATA / ("womd_training_smoke.npz" if SMOKE else "womd_training_paper.npz")
cmd = [sys.executable, str(ROOT / "scripts/01_build_official_womd_samples.py"),
       *map(str, sorted(train_dir.glob("training.tfrecord-*"))),
       "--output", str(train_npz), "--max-vehicles", "16"]
if SMOKE:
    cmd += ["--max-scenarios", "200"]
subprocess.run(cmd, cwd=ROOT, check=True)
assert train_npz.exists() and train_npz.stat().st_size > 0
print("Created training samples:", train_npz, train_npz.stat().st_size, "bytes")


In [ ]:
# 6. Validate the immutable four-objective × three-seed experiment plan
seeds = ["20260827", "20260828", "20260829"]
epochs = "2" if SMOKE else "80"
run_out = DRIVE_OUT / ("smoke_ablation" if SMOKE else "paper_ablation")
subprocess.run([sys.executable, str(ROOT / "scripts/04_run_training_ablation.py"),
                str(train_npz), "--output", str(run_out), "--epochs", epochs,
                "--seeds", *seeds, "--plan-only"], cwd=ROOT, check=True)
print("Experiment plan validated:", run_out)


In [ ]:
# 7. Run/resume learned ablations; completed objective/seed runs are preserved
subprocess.run([sys.executable, str(ROOT / "scripts/04_run_training_ablation.py"),
                str(train_npz), "--output", str(run_out), "--epochs", epochs,
                "--batch-size", "32", "--seeds", *seeds,
                "--lambda-link", "0.2", "--lambda-outage", "0.1"],
               cwd=ROOT, check=True)
print("Training ablation completed:", run_out)


In [ ]:
# 8. Verify outputs and create a small persistent archive (TFRecords are excluded)
result_files = sorted(run_out.rglob("training_result.json"))
checkpoint_files = sorted(run_out.rglob("*.pt"))
print("Training result files:", len(result_files))
print("Checkpoint files:", len(checkpoint_files))

expected_runs = 4 * len(seeds)
if len(result_files) < expected_runs:
    raise RuntimeError(f"Only {len(result_files)}/{expected_runs} runs completed. Re-run cell 7 to resume.")

archive = DRIVE_OUT / ("womd_smoke_artifacts.tar.gz" if SMOKE else "womd_paper_artifacts.tar.gz")
subprocess.run(["tar", "-czf", str(archive), "-C", str(DRIVE_OUT), run_out.name], check=True)
print("SUCCESS")
print("Saved archive:", archive)
print("Next action:" if SMOKE else "Paper training complete:",
      'set RUN_MODE = "paper" and run cells 2-8' if SMOKE else archive)
